In [ ]:
!nvidia-smi

In [ ]:
%%writefile naive_gemm.cuh
__global__ void naiveGemm(float *A, float *B, float *C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < M && col < N) {
        float sum = 0.f;
        for (int k = 0; k < K; k++)
            sum += A[row * K + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

In [ ]:
%%writefile naive_gemm.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include "naive_gemm.cuh"

int main() {
    int M = 1024, K = 1024, N = 1024;
    float *A, *B, *C;
    cudaMalloc(&A, M*K*4); cudaMalloc(&B, K*N*4); cudaMalloc(&C, M*N*4);
    dim3 blk(16, 16), grd((N+15)/16, (M+15)/16);
    naiveGemm<<<grd, blk>>>(A, B, C, M, K, N);
    cudaDeviceSynchronize();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int reps = 10; cudaEventRecord(s);
    for (int i = 0; i < reps; i++) naiveGemm<<<grd, blk>>>(A, B, C, M, K, N);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms, s, e);
    printf("Naive GEMM: %.2f TFLOPS  %.2f ms/iter\n",
           2.0*M*K*N*reps/(ms*1e9), ms/reps);
    cudaFree(A); cudaFree(B); cudaFree(C);
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o naive_gemm naive_gemm.cu && ./naive_gemm